# 11.34 Self-play & population-based training

Self-play creates its own curriculum by making yesterday’s policy today’s opponent.

Reinforcement learning is where a prediction changes what data arrives next. Probability supplies transition probabilities and expectations; optimization supplies iterative improvement. These notebooks keep the environments tiny and CPU-only while implementing the real decision rule. Save a copy to Drive to edit.

In [ ]:

import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

SEED = 1135
random.seed(SEED)
np.random.seed(SEED)

ACTIONS = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])
ACTION_NAMES = ["up", "right", "down", "left"]


@dataclass
class GridEnv:
    name: str
    height: int
    width: int
    start: tuple
    goal: tuple
    hazards: set
    traps: set
    slip: float
    wind: float
    budget: float
    max_steps: int
    reward_goal: float = 5.0
    reward_step: float = -0.05
    cost_hazard: float = 1.0

    @property
    def n_states(self):
        return self.height * self.width

    @property
    def n_actions(self):
        return 4

    def state_to_pos(self, state):
        row = state // self.width
        col = state % self.width
        return (row, col)

    def pos_to_state(self, pos):
        return pos[0] * self.width + pos[1]

    def in_bounds(self, pos):
        row, col = pos
        return 0 <= row < self.height and 0 <= col < self.width

    def start_state(self):
        return self.pos_to_state(self.start)

    def goal_state(self):
        return self.pos_to_state(self.goal)

    def transition(self, state, action, slip_override=None):
        slip = self.slip if slip_override is None else slip_override
        probs = []
        primary = self._move(state, action)
        probs.append((1.0 - slip, primary))
        side_actions = [(action + 1) % 4, (action - 1) % 4]
        for side in side_actions:
            probs.append((slip / 2.0, self._move(state, side)))
        merged = {}
        for prob, next_state in probs:
            merged[next_state] = merged.get(next_state, 0.0) + prob
        return list(merged.items())

    def _move(self, state, action):
        if state == self.goal_state():
            return state
        pos = np.array(self.state_to_pos(state))
        next_pos = tuple(pos + ACTIONS[action])
        if self.wind > 0 and action == 1:
            next_pos = (max(0, next_pos[0] - 1), next_pos[1])
        if not self.in_bounds(next_pos):
            next_pos = tuple(pos)
        return self.pos_to_state(next_pos)

    def reward_cost_done(self, state, action, next_state):
        pos = self.state_to_pos(next_state)
        done = next_state == self.goal_state()
        reward = self.reward_goal if done else self.reward_step
        cost = self.cost_hazard if pos in self.hazards or pos in self.traps else 0.0
        if pos in self.traps:
            reward -= 1.0
        return reward, cost, done


def make_f12_ladder():
    return [
        GridEnv("D1 two-state chain", 1, 2, (0, 0), (0, 1), {(0, 1)}, set(), 0.00, 0.0, 1.0, 3),
        GridEnv("D2 slippery 3-state hazards", 1, 3, (0, 0), (0, 2), {(0, 1)}, set(), 0.10, 0.0, 1.0, 6),
        GridEnv("D3 4x4 gridworld hazards", 4, 4, (3, 0), (0, 3), {(2, 1), (1, 2)}, set(), 0.08, 0.0, 1.2, 18),
        GridEnv("D4 stochastic windy grid", 5, 5, (4, 0), (0, 4), {(3, 1), (2, 2), (1, 3)}, set(), 0.15, 0.25, 1.4, 28),
        GridEnv("D5 sparse grid with traps", 6, 6, (5, 0), (0, 5), {(4, 1), (3, 2), (2, 3)}, {(1, 4), (4, 4)}, 0.18, 0.30, 1.5, 40),
    ]


def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)


def discounted_return(rewards, gamma=0.9):
    total = 0.0
    for step, reward in enumerate(rewards):
        total += (gamma ** step) * reward
    return total


def value_iteration(env, gamma=0.9, penalty=0.0, slip_override=None, iterations=80):
    values = np.zeros(env.n_states)
    q_values = np.zeros((env.n_states, env.n_actions))
    for _ in range(iterations):
        new_values = values.copy()
        for state in range(env.n_states):
            if state == env.goal_state():
                continue
            action_scores = []
            for action in range(env.n_actions):
                score = 0.0
                for prob, next_state in env.transition(state, action, slip_override):
                    reward, cost, done = env.reward_cost_done(state, action, next_state)
                    bootstrap = 0.0 if done else gamma * values[next_state]
                    score += prob * (reward - penalty * cost + bootstrap)
                action_scores.append(score)
            new_values[state] = max(action_scores)
            q_values[state] = action_scores
        values = new_values
    policy = np.argmax(q_values, axis=1)
    return values, q_values, policy


def evaluate_policy(env, policy, gamma=0.9, episodes=40, seed=0, slip_override=None):
    rng = np.random.default_rng(seed)
    returns = []
    costs = []
    wins = []
    paths = []
    for episode in range(episodes):
        state = env.start_state()
        rewards = []
        cost_values = []
        path = [state]
        for step in range(env.max_steps):
            action = int(policy[state])
            transitions = env.transition(state, action, slip_override)
            probs = np.array([item[0] for item in transitions])
            idx = rng.choice(len(transitions), p=probs)
            next_state = transitions[idx][1]
            reward, cost, done = env.reward_cost_done(state, action, next_state)
            rewards.append(reward)
            cost_values.append(cost)
            state = next_state
            path.append(state)
            if done:
                break
        returns.append(discounted_return(rewards, gamma))
        costs.append(sum(cost_values))
        wins.append(float(state == env.goal_state()))
        paths.append(path)
    return {
        "return": float(np.mean(returns)),
        "cost": float(np.mean(costs)),
        "win_rate": float(np.mean(wins)),
        "path": paths[0],
    }



def preview_ladder(ladder):
    rows = []
    for idx, env in enumerate(ladder, start=1):
        rows.append({
            "rung": f"D{idx}",
            "name": env.name,
            "shape": (env.height, env.width),
            "states": env.n_states,
            "slip": env.slip,
            "budget": env.budget,
            "hazards": len(env.hazards) + len(env.traps),
        })
    return rows


def plot_env_panel(ax, env, values, title):
    image = np.asarray(values).reshape(env.height, env.width)
    ax.imshow(image, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    for hazard in env.hazards:
        ax.text(hazard[1], hazard[0], "H", color="white", ha="center", va="center")
    for trap in env.traps:
        ax.text(trap[1], trap[0], "T", color="red", ha="center", va="center")
    ax.text(env.start[1], env.start[0], "S", color="white", ha="center", va="center")
    ax.text(env.goal[1], env.goal[0], "G", color="white", ha="center", va="center")


## The concept, built once (D1)

Self-play still evaluates $$V^\pi(s)=\sum_a\pi(a\mid s)\sum_{s'}P(s'\mid s,a)(R(s,a,s')+\gamma V^\pi(s'))$$, but the opponent policy is part of the environment. Exact lesson arithmetic comes first.

In [ ]:

rewards = [1, 0, 2]
gamma = 0.9
G = discounted_return(rewards, gamma)
y = 1 + gamma * 0.8
q_new = 0.4 + 0.5 * (y - 0.4)
probs = softmax([1, 0])
expected_reward = probs[0] * 2 + probs[1] * 0
ucb = 0.55 + math.sqrt(2 * math.log(20) / 5)
print("G", round(G, 3))
print("target", round(y, 3))
print("Q_new", round(q_new, 3))
print("policy", np.round(probs, 3))
print("expected reward", round(float(expected_reward), 3))
print("UCB", round(ucb, 3))
assert round(G, 3) == 2.620
assert round(y, 3) == 1.720
assert round(q_new, 3) == 1.060
assert round(float(probs[0]), 3) == 0.731
assert round(float(probs[1]), 3) == 0.269
assert round(float(expected_reward), 3) == 1.462
assert round(ucb, 3) == 1.645


On D1, a two-action game updates against the previous opponent, then population replay averages several opponents to reduce non-stationarity.

In [ ]:
def matrix_payoff(level):
    base = np.array([[0.0, -1.0], [1.0, 0.0]])
    if level >= 3:
        base = np.array([[0.2, -1.0], [1.0, -0.1]])
    if level >= 5:
        base = np.array([[0.3, -1.2], [1.1, -0.2]])
    return base


def expected_payoff(policy, opponent, payoff):
    return float(policy @ payoff @ opponent)


def self_play_update(logits, opponent_policy, payoff, lr=0.8):
    policy = softmax(logits)
    action_values = payoff @ opponent_policy
    baseline = float(policy @ action_values)
    advantage = action_values - baseline
    new_logits = logits + lr * advantage
    new_policy = softmax(new_logits)
    return new_logits, new_policy, baseline


def train_self_play_population(level=1, generations=20, population_size=4, use_population=True):
    payoff = matrix_payoff(level)
    logits = np.zeros(2)
    population = [softmax(logits)]
    history = []
    for generation in range(generations):
        if use_population:
            opponent = np.mean(population, axis=0)
        else:
            opponent = population[-1]
        logits, policy, score = self_play_update(logits, opponent, payoff)
        win_rate = 1.0 / (1.0 + np.exp(-score))
        history.append({"generation": generation, "return": score, "win_rate": float(win_rate), "policy_1": float(policy[1])})
        population.append(policy)
        population = population[-population_size:]
    return policy, population, history, payoff


policy, population, history, payoff = train_self_play_population(level=1, generations=6)
print("payoff", payoff)
print("policy", np.round(policy, 3))
print("last", history[-1])
assert len(population) <= 4

## The dataset ladder

Family F12 uses a D1–D5 sequential-decision ladder inline: a two-state chain, slippery chain, 4x4 grid, windy grid, and sparse trap grid.

In [ ]:
ladder = make_f12_ladder()
for row in preview_ladder(ladder):
    print(row)
print("sample D5 policy grid shape", (ladder[-1].height, ladder[-1].width))

## Run the SAME method across D1-D5

Apply the same method to every rung and collect the plan metric.

In [ ]:
results = []
artifacts = []
for index, env in enumerate(ladder, start=1):
    level = index
    policy, population, history, payoff = train_self_play_population(level=level, generations=22, use_population=True)
    latest_policy, latest_population, latest_history, latest_payoff = train_self_play_population(level=level, generations=22, use_population=False)
    row = {
        "rung": f"D{index}",
        "return": history[-1]["return"],
        "win_rate": history[-1]["win_rate"],
        "latest_win_rate": latest_history[-1]["win_rate"],
        "policy_1": history[-1]["policy_1"],
    }
    results.append(row)
    artifacts.append((env, policy, population, history, payoff))
print("rung | return | win_rate population | win_rate latest-only | action1_prob")
for row in results:
    print(row["rung"], round(row["return"], 3), round(row["win_rate"], 3), round(row["latest_win_rate"], 3), round(row["policy_1"], 3))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, (env, policy, population, history, payoff) in enumerate(artifacts):
    policy_grid_values = np.full((env.height, env.width), policy[1])
    plot_env_panel(axes[0, col], env, policy_grid_values, f"D{col + 1} policy")
    axes[1, col].plot([item["win_rate"] for item in history])
    axes[1, col].set_ylim(0, 1)
    axes[1, col].set_title(f"D{col + 1} win-rate")
    axes[1, col].set_xlabel("generation")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
xs = np.arange(1, 6)
plt.plot(xs, [row["win_rate"] for row in results], marker="o", label="population")
plt.plot(xs, [row["latest_win_rate"] for row in results], marker="s", label="latest only")
plt.xticks(xs, [row["rung"] for row in results])
plt.xlabel("rung")
plt.ylabel("win-rate")
plt.title("Self-play win-rate across D1-D5")
plt.legend()
plt.show()

## Pitfall on D5: bootstrapping from a moving target

Training only against the newest opponent can chase a moving target and forget older strategies. Population replay stabilizes the opponent distribution.

In [ ]:
latest_policy, latest_population, latest_history, latest_payoff = train_self_play_population(level=5, generations=30, use_population=False)
pop_policy, pop_population, pop_history, pop_payoff = train_self_play_population(level=5, generations=30, use_population=True)
print("latest-only final", latest_history[-1])
print("population final", pop_history[-1])
print("win-rate lift", round(pop_history[-1]["win_rate"] - latest_history[-1]["win_rate"], 3))

## Evaluate it + Practice

- Compare the reported return / win-rate against a no-skill baseline such as a random or immediate-reward policy.
- Sanity check that the D1 result matches the exact lesson arithmetic before trusting harder rungs.
- Ablate the key idea, such as removing constraints, randomization, return conditioning, population replay, or search.
- Watch failure signals: budget violations, transfer collapse, unsupported target returns, non-stationary opponents, or shallow reward chasing.

Practice:
1. Change the discount from 0.9 to 0.8 and predict which rung changes most.


2. Add one hazard or trap to D4 and rerun only the small table, not a long training job.

3. Replace the no-skill baseline with a hand-written safe policy and compare the metric.